In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import umap
from IPython.display import Image, display

working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir.parent if working_dir.name == "figures" else working_dir
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGUTIL_DIR = FIGURES_DIR / "figutil"
FIGDATA_DIR = FIGURES_DIR / "figdata/token_umap"
FIGDATA_DIR.mkdir(parents=True, exist_ok=True)

for module_dir in (PROJECT_ROOT, FIGUTIL_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from plot_current_checkpoint_token_umap_legacy_exact import (
    DEFAULT_CHECKPOINT,
    DEFAULT_FREQUENCY_DATA,
    DEFAULT_LABELS,
    DEFAULT_NOTEBOOK,
    MEDICATION_TOKEN_IDS,
    compact_inset_configs,
    configure_exact_legacy_font,
    exact_plot_functions,
    load_frequencies,
    load_metadata,
)
from utils import load_checkpoint

CHECKPOINT_PATH = DEFAULT_CHECKPOINT
LABELS_PATH = DEFAULT_LABELS
FREQUENCY_DATA_PATH = DEFAULT_FREQUENCY_DATA
LEGACY_NOTEBOOK_PATH = DEFAULT_NOTEBOOK

for input_path in (
    CHECKPOINT_PATH,
    LABELS_PATH,
    FREQUENCY_DATA_PATH,
    LEGACY_NOTEBOOK_PATH,
):
    if not input_path.is_file():
        raise FileNotFoundError(input_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Figure data: {FIGDATA_DIR}")
print("Figure file export: disabled until final approval")

In [ ]:
font_path, font_name = configure_exact_legacy_font()
token_meta, legend_info = load_metadata(LABELS_PATH)
token_counts, frequency_rows = load_frequencies(FREQUENCY_DATA_PATH)
draw_main, draw_clean, renderer_hashes = exact_plot_functions(LEGACY_NOTEBOOK_PATH)

print(f"Metadata tokens: {len(token_meta):,}")
print(f"Training rows used for frequency: {frequency_rows:,}")
print(f"Font: {font_name}")

In [ ]:
model, checkpoint = load_checkpoint(CHECKPOINT_PATH, "cpu")
valid_token_ids = sorted(
    token_id
    for token_id in token_meta
    if 0 <= token_id < model.config.vocab_size
)

embeddings = (
    model.token.weight.detach().cpu().numpy()[valid_token_ids]
    .astype(np.float32, copy=True)
)

assert embeddings.shape == (1267, 384), embeddings.shape
assert np.isfinite(embeddings).all()

print(f"Checkpoint step: {int(checkpoint['step']):,}")
print(f"Token embedding matrix: {embeddings.shape}")

In [ ]:
reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.02,
    metric="cosine",
    random_state=42,
    n_jobs=1,
)
coordinates = reducer.fit_transform(embeddings).astype(np.float32, copy=False)
# Rotate 90° counter-clockwise: (x, y) -> (-y, x)
coordinates = np.column_stack((-coordinates[:, 1], coordinates[:, 0])).astype(
    np.float32, copy=False
)
assert coordinates.shape == (1267, 2)
assert np.isfinite(coordinates).all()

coordinate_table = pd.DataFrame(
    {
        "x": coordinates[:, 0],
        "y": coordinates[:, 1],
        "token_id": valid_token_ids,
        "name": [token_meta[token_id]["name"] for token_id in valid_token_ids],
        "chapter_short": [
            token_meta[token_id]["chapter_short"] for token_id in valid_token_ids
        ],
        "count": [token_counts.get(token_id, 0) for token_id in valid_token_ids],
    }
)

coordinates_path = FIGDATA_DIR / "umap_coordinates.csv"
coordinate_table.to_csv(coordinates_path, index=False)
print(f"Saved coordinates: {coordinates_path}")

In [ ]:
MAIN_MIN_EVENT_COUNT = 100
display_token_counts = token_counts.copy()
for token_id in MEDICATION_TOKEN_IDS:
    display_token_counts[token_id] = max(
        display_token_counts.get(token_id, 0), MAIN_MIN_EVENT_COUNT
    )

# Insets are defined relative to token positions in data space, so rotate
# their selection radii/offsets to match the rotated coordinates.
# figure_bounds / description_bounds reserve compact slots on the canvas.
inset_configs = compact_inset_configs()
for cfg in inset_configs:
    if "x_radius" in cfg and "y_radius" in cfg:
        cfg["x_radius"], cfg["y_radius"] = cfg["y_radius"], cfg["x_radius"]
    if "center_offset" in cfg:
        dx, dy = cfg["center_offset"]
        cfg["center_offset"] = (-dy, dx)
    if "include_pad_x" in cfg and "include_pad_y" in cfg:
        cfg["include_pad_x"], cfg["include_pad_y"] = cfg["include_pad_y"], cfg["include_pad_x"]

main_figure = draw_main(
    coordinates,
    valid_token_ids,
    token_meta,
    display_token_counts,
    legend_info,
    target_label_ids=set(MEDICATION_TOKEN_IDS),
    figsize=(11.4, 6.8),
    min_event_count=MAIN_MIN_EVENT_COUNT,
    inset_configs=inset_configs,
)

# Figure export is intentionally disabled until the final figure is approved.
# main_figure.savefig(FIGURES_DIR / "umap.png", dpi=900, bbox_inches="tight", pad_inches=0.08)
# main_figure.savefig(FIGURES_DIR / "umap.pdf", dpi=900, bbox_inches="tight", pad_inches=0.08)
display(main_figure)
plt.close(main_figure)
print("Rendered inline; figure files were not saved.")

In [ ]:
CLEAN_MIN_EVENT_COUNT = 100
clean_figure = draw_clean(
    coordinates,
    valid_token_ids,
    token_meta,
    token_counts,
    figsize=(5, 5),
    min_event_count=CLEAN_MIN_EVENT_COUNT,
)

# Figure export is intentionally disabled until the final figure is approved.
# clean_figure.savefig(FIGURES_DIR / "umap_clean.png", dpi=900, transparent=True, pad_inches=0.0)
# clean_figure.savefig(FIGURES_DIR / "umap_clean.pdf", transparent=True, pad_inches=0.0)
display(clean_figure)
plt.close(clean_figure)
print("Rendered inline; figure files were not saved.")

## 7. Checks

Medication token 10개가 모두 좌표에 포함되고 메인 패널의 표시 대상으로 유지되는지 확인합니다.


In [ ]:
medication_check = (
    coordinate_table[
        coordinate_table["token_id"].isin(MEDICATION_TOKEN_IDS)
    ][["token_id", "name", "count", "x", "y"]]
    .sort_values("token_id")
    .reset_index(drop=True)
)

assert medication_check["token_id"].tolist() == list(MEDICATION_TOKEN_IDS)
assert len(medication_check) == 10

main_display_mask = (
    (coordinate_table["count"] >= MAIN_MIN_EVENT_COUNT)
    | coordinate_table["token_id"].isin(MEDICATION_TOKEN_IDS)
)

summary = pd.Series(
    {
        "UMAP tokens": len(coordinate_table),
        "Main displayed tokens": int(main_display_mask.sum()),
        "Clean displayed tokens": int((coordinate_table["count"] >= CLEAN_MIN_EVENT_COUNT).sum()),
        "Medication tokens displayed": len(medication_check),
    },
    name="value",
).to_frame()

display(summary)
display(medication_check.round({"x": 3, "y": 3}))


## Takeaways

- UMAP fitting에는 질환 1,256개, diabetes medication 10개, Death 1개 등 총 1,267개 token이 사용됩니다.
- 메인 figure에서는 질환에 count ≥ 100을 적용하면서 medication 10개는 모두 유지합니다.
- GLP-1, SGLT-2 inhibitor, Other처럼 빈도가 25 미만인 medication도 작은 점과 이름으로 확인할 수 있습니다.
- Diabetes-related conditions inset은 기존 legacy 버전보다 화면 점유 면적과 선택 반경이 축소되어 있습니다.
